In [8]:
project_root = "/home/krzysztof/studia/magisterka/time-series-invariance"

In [9]:
import os
subdir = ""
os.chdir(os.path.join(project_root, subdir))

import numpy as np

from src.data_generation.simple_data_generation import shift_time_series, shrunk_time_series
from notebooks.evaluations.helpers import (
        apply_embedding_function_shrunk,
        apply_embedding_function_shifts,
        load_dataset,
        load_distortions,
    )

In [10]:
data_dir = "data/datasets"
dataset_names = [name[:-4] for name in os.listdir(data_dir)]
dataset_names

['CBF',
 'OSULeaf',
 'Adiac',
 'Fish',
 'MedicalImages',
 'Wafer',
 'SwedishLeaf',
 'FacesUCR',
 'SyntheticControl',
 'FaceFour',
 'Trace',
 'Beef',
 'GunPoint',
 'TwoPatterns',
 'Coffee',
 'OliveOil',
 'ECG200']

In [18]:
subdir = "ts2vec"
os.chdir(os.path.join(project_root, subdir))

from ts2vec.ts2vec import TS2Vec

print("Model loaded successfully!")

subdir = ""
os.chdir(os.path.join(project_root, subdir))

Model loaded successfully!


In [ ]:
def tf2vec_embedding_function(a: np.ndarray, model) -> np.ndarray:
    a = a[None, ..., None]
    a = np.repeat(a, 14, axis=2)
    v = model.encode(a, encoding_window='full_series')
    return v.reshape(-1)

In [ ]:
import re
pattern = re.compile(r"(.+)__test_run_(\d{8})(\d{6})")

model_paths = {}
model_dir = "ts2vec/training"
for name in os.listdir(model_dir):
    match = pattern.match(name)
    if match:
        dataset_name = match.group(1)
        date = float(match.group(2)) + float(match.group(3))/1e6,
        if date > model_paths.get(dataset_name, {}).get("date", 0):
            model_paths[dataset_name] = {
                "date": date,
                "path": os.path.join(model_dir, name, "model.pkl")
            }
model_paths

{'TwoPatterns': 'ts2vec/training/TwoPatterns__test_run_20260316_105903/model.pkl',
 'GunPoint': 'ts2vec/training/GunPoint__test_run_20260316_105853/model.pkl',
 'CBF': 'ts2vec/training/CBF__test_run_20260316_105552/model.pkl',
 'Adiac': 'ts2vec/training/Adiac__test_run_20260316_105646/model.pkl',
 'OSULeaf': 'ts2vec/training/OSULeaf__test_run_20260316_105607/model.pkl',
 'Coffee': 'ts2vec/training/Coffee__test_run_20260316_105919/model.pkl',
 'OliveOil': 'ts2vec/training/OliveOil__test_run_20260316_105927/model.pkl',
 'SwedishLeaf': 'ts2vec/training/SwedishLeaf__test_run_20260316_105742/model.pkl',
 'ECG200': 'ts2vec/training/ECG200__test_run_20260316_105935/model.pkl',
 'FaceFour': 'ts2vec/training/FaceFour__test_run_20260316_105826/model.pkl',
 'MedicalImages': 'ts2vec/training/MedicalImages__test_run_20260316_105713/model.pkl',
 'Trace': 'ts2vec/training/Trace__test_run_20260316_105835/model.pkl',
 'FacesUCR': 'ts2vec/training/FacesUCR__test_run_20260316_105757/model.pkl',
 'Wafer':

In [22]:
distortion_type = "shrink"
for dataset_name in dataset_names:
    data = load_dataset(dataset_name)
    print(f"data shape: {data.shape}")
    random_shrunks = load_distortions(dataset_name, distortion_type)

    model_path = model_paths[dataset_name]
    model = TS2Vec(input_dims=1, device='cpu', output_dims=320)
    model.load(model_path)

    embeddings = apply_embedding_function_shifts(data, tf2vec_embedding_function, random_shrunks)
    with open(f"notebooks/evaluations/embeddings/ts2vec/{distortion_type}/{dataset_name}.npy", "wb") as f:
        np.save(f, embeddings)


data shape: (500, 128)


RuntimeError: Error(s) in loading state_dict for AveragedModel:
	size mismatch for module.input_fc.weight: copying a param with shape torch.Size([64, 128]) from checkpoint, the shape in current model is torch.Size([64, 1]).